# Statistical Validation for Time-Shifted A/B Testing

This notebook is intentionally separate from the Ansible pipeline and focuses on statistical validity of the time-shifted A/B experiment.

## Primary claims
1. **Transition reliability**: effective switch latency is acceptable (`<= FLIP_SLA_SECONDS`, default 30s).
2. **Service continuity**: throughput dip during transition is acceptable (`<= CONTINUITY_MAX_DIP`, default 10%).

## Effective flip semantics
- A flip is considered **effective** at the first timestamp where `from_version <= INACTIVE_DROP_THRESHOLD` (default `0.0`), for `EFFECTIVE_CONSECUTIVE_SAMPLES` consecutive samples.
- **Switch latency** is measured from apply timestamp to effective timestamp.
- Transition continuity is evaluated around apply timestamp with pre/post windows (`BASELINE_PRE_SECONDS`, `POST_FLIP_WINDOW_SECONDS`).

The workflow auto-discovers archived and latest artifacts, applies run/flip QC, and writes thesis-ready statistical outputs.

## Run collection protocol

1. Execute **20 full runs**.
2. Randomize start version by alternating versions order per run (target: 10 runs start with v1, 10 with v2).
3. Archive each run in `Tests/istio-ab-testing/YYYYMMDD` with:
   - `istio_tcp_<utc>.json`
   - `flip_events_<utc>.json`
   - optional `istio_tcp_<utc>.csv` / `istio_tcp_<utc>.html`
4. Re-run this notebook after each batch; QC-failed runs/flips are excluded and documented.

## Output artifacts
- `Tests/istio-ab-testing/analysis/stat_results_<utc>.csv`
- `Tests/istio-ab-testing/analysis/routing_and_latency_<utc>.html`
- `Tests/istio-ab-testing/analysis/statistical_conclusion_<utc>.md`


In [ ]:
# Parameters (single configuration cell)
from pathlib import Path
from datetime import datetime, timezone


def _find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "Experiments").exists() and (candidate / "Tests").exists():
            return candidate
    return start


def _dur_to_seconds(duration: str) -> int:
    txt = str(duration).strip().lower()
    if txt.endswith("ms"):
        return max(1, int(float(txt[:-2]) / 1000.0))
    if txt.endswith("s"):
        return int(float(txt[:-1]))
    if txt.endswith("m"):
        return int(float(txt[:-1]) * 60)
    if txt.endswith("h"):
        return int(float(txt[:-1]) * 3600)
    return int(float(txt))


PROJECT_ROOT = _find_project_root(Path.cwd())
DATA_ROOTS = [
    PROJECT_ROOT / "Tests/istio-ab-testing",
    PROJECT_ROOT / "Experiments/istio-ab-testing/artifacts/metrics",
]
OUTPUT_DIR = PROJECT_ROOT / "Tests/istio-ab-testing/analysis"

ALPHA = 0.05
PAIR_TOLERANCE_SECONDS = 10
MIN_RUNS_FOR_CONFIRMATORY = 12
TARGET_RUNS = 20
N_BOOT = 10_000

PROM_STEP = "15s"
PROM_RANGE_WINDOW = "30s"
PROM_STEP_SECONDS = _dur_to_seconds(PROM_STEP)
PROM_RANGE_WINDOW_SECONDS = _dur_to_seconds(PROM_RANGE_WINDOW)

# Effective switch detection
INACTIVE_DROP_THRESHOLD = 0.0
EFFECTIVE_CONSECUTIVE_SAMPLES = 1

# Primary thresholds
FLIP_SLA_SECONDS = 30
CONTINUITY_MAX_DIP = 0.10

# Continuity windows around each flip apply timestamp
BASELINE_PRE_SECONDS = 30
POST_FLIP_WINDOW_SECONDS = 30
RECOVERY_TARGET_RATIO = 0.95

# Secondary exploratory throughput window, anchored after effective switch
STABILIZATION_SECONDS = max(45, PROM_RANGE_WINDOW_SECONDS, 2 * PROM_STEP_SECONDS)

EXPECTED_FLIP_EVENTS = 8

ANALYSIS_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOTS:", [str(p) for p in DATA_ROOTS])
print("OUTPUT_DIR:", OUTPUT_DIR)
print("INACTIVE_DROP_THRESHOLD:", INACTIVE_DROP_THRESHOLD)
print("EFFECTIVE_CONSECUTIVE_SAMPLES:", EFFECTIVE_CONSECUTIVE_SAMPLES)
print("FLIP_SLA_SECONDS:", FLIP_SLA_SECONDS)
print("CONTINUITY_MAX_DIP:", CONTINUITY_MAX_DIP)
print("BASELINE_PRE_SECONDS:", BASELINE_PRE_SECONDS)
print("POST_FLIP_WINDOW_SECONDS:", POST_FLIP_WINDOW_SECONDS)
print("RECOVERY_TARGET_RATIO:", RECOVERY_TARGET_RATIO)
print("STABILIZATION_SECONDS:", STABILIZATION_SECONDS)
print("ANALYSIS_TIMESTAMP_UTC:", ANALYSIS_TIMESTAMP_UTC)


In [ ]:
# Environment/package check (fail fast)
import importlib.util

required = ["jupyter", "ipykernel", "pandas", "numpy", "scipy", "statsmodels", "plotly"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    raise RuntimeError(
        "Missing required packages: "
        + ", ".join(missing)
        + "\nInstall with one of:\n"
        + "  uv pip install jupyter ipykernel pandas numpy scipy statsmodels plotly\n"
        + "  python3 -m pip install jupyter ipykernel pandas numpy scipy statsmodels plotly"
    )

print("All required packages are available.")


In [ ]:
# Imports
import json
import math
from datetime import timedelta

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display


In [ ]:
# Discovery and pairing helpers

def parse_iso_utc(value):
    if value is None:
        return pd.NaT
    if isinstance(value, (int, float)):
        return pd.to_datetime(float(value), unit="s", utc=True, errors="coerce")
    text = str(value).strip()
    if not text:
        return pd.NaT
    if text.endswith("Z"):
        text = text[:-1] + "+00:00"
    return pd.to_datetime(text, utc=True, errors="coerce")


def parse_suffix_datetime(path_or_name, prefix):
    name = Path(path_or_name).name
    if not name.startswith(prefix):
        return pd.NaT
    suffix = name[len(prefix):]
    if suffix.endswith(".json"):
        suffix = suffix[:-5]
    elif suffix.endswith(".csv"):
        suffix = suffix[:-4]
    elif suffix.endswith(".html"):
        suffix = suffix[:-5]
    return parse_iso_utc(suffix)


def _store_latest(mapping, key, path):
    existing = mapping.get(key)
    if existing is None or path.stat().st_mtime > existing.stat().st_mtime:
        mapping[key] = path


def discover_artifact_files(data_roots):
    metrics_json = {}
    metrics_csv = {}
    flips_json = {}
    discovery_notes = []

    for root in data_roots:
        if not root.exists():
            discovery_notes.append(f"Missing data root: {root}")
            continue

        for path in root.rglob("istio_tcp_*.json"):
            key = path.stem.replace("istio_tcp_", "")
            _store_latest(metrics_json, key, path)

        for path in root.rglob("istio_tcp_*.csv"):
            key = path.stem.replace("istio_tcp_", "")
            _store_latest(metrics_csv, key, path)

        for path in root.rglob("flip_events_*.json"):
            key = path.stem.replace("flip_events_", "")
            _store_latest(flips_json, key, path)

    return {
        "metrics_json": metrics_json,
        "metrics_csv": metrics_csv,
        "flips_json": flips_json,
        "notes": discovery_notes,
    }


def pair_metrics_and_flips(artifacts, tolerance_seconds=10):
    metrics_json = artifacts["metrics_json"]
    metrics_csv = artifacts["metrics_csv"]
    flips_json = artifacts["flips_json"]

    candidate_suffixes = sorted(set(metrics_json.keys()) | set(metrics_csv.keys()))

    flip_ts = {key: parse_iso_utc(key) for key in flips_json.keys()}
    flip_ts = {k: v for k, v in flip_ts.items() if not pd.isna(v)}

    pair_rows = []
    pair_qc_rows = []

    for suffix in candidate_suffixes:
        run_ts = parse_iso_utc(suffix)
        if pd.isna(run_ts):
            pair_qc_rows.append({
                "run_id": suffix,
                "included_for_stats": False,
                "reason": "Unparseable metrics timestamp suffix",
            })
            continue

        metric_json_path = metrics_json.get(suffix)
        metric_csv_path = metrics_csv.get(suffix)

        nearest_flip_suffix = None
        nearest_flip_delta = np.nan
        if flip_ts:
            nearest_flip_suffix, nearest_flip_time = min(
                flip_ts.items(),
                key=lambda kv: abs((kv[1] - run_ts).total_seconds()),
            )
            nearest_flip_delta = abs((nearest_flip_time - run_ts).total_seconds())

        pair_valid = (
            nearest_flip_suffix is not None
            and nearest_flip_delta <= tolerance_seconds
        )

        pair_rows.append({
            "run_id": suffix,
            "run_ts": run_ts,
            "metric_json_path": str(metric_json_path) if metric_json_path else "",
            "metric_csv_path": str(metric_csv_path) if metric_csv_path else "",
            "flip_json_path": str(flips_json.get(nearest_flip_suffix, "")) if pair_valid else "",
            "nearest_flip_suffix": nearest_flip_suffix,
            "pair_delta_seconds": nearest_flip_delta,
            "pair_valid": pair_valid,
        })

        if not pair_valid:
            reason = "No flip events file within tolerance"
            if nearest_flip_suffix is not None:
                reason = f"Nearest flip file outside tolerance ({nearest_flip_delta:.1f}s > {tolerance_seconds}s)"
            pair_qc_rows.append({
                "run_id": suffix,
                "included_for_stats": False,
                "reason": reason,
            })

    pairs_df = pd.DataFrame(pair_rows)
    pair_qc_df = pd.DataFrame(pair_qc_rows)
    return pairs_df, pair_qc_df


In [ ]:
# Run-level loading and per-flip feature engineering

def load_metrics_timeseries(metric_json_path, metric_csv_path):
    load_notes = []

    if metric_json_path:
        path = Path(metric_json_path)
        if path.exists():
            try:
                with path.open(encoding="utf-8") as handle:
                    payload = json.load(handle)
                rows = []
                for result in payload.get("data", {}).get("result", []):
                    metric = result.get("metric", {}) or {}
                    version = metric.get("destination_version") or metric.get("version") or "unknown"
                    samples = result.get("values")
                    if not samples and result.get("value"):
                        samples = [result["value"]]
                    for sample in samples or []:
                        if not sample or len(sample) < 2:
                            continue
                        ts = parse_iso_utc(sample[0])
                        if pd.isna(ts):
                            continue
                        try:
                            value = float(sample[1])
                        except Exception:
                            continue
                        rows.append({"time": ts, "version": version, "value": value})

                if rows:
                    df = pd.DataFrame(rows)
                    pivot = (
                        df.pivot_table(index="time", columns="version", values="value", aggfunc="mean")
                        .sort_index()
                        .fillna(0.0)
                    )
                    pivot.columns = [str(c) for c in pivot.columns]
                    return pivot, "json", load_notes
                load_notes.append("JSON metrics contained no usable rows")
            except Exception as exc:
                load_notes.append(f"JSON load failed: {exc}")

    if metric_csv_path:
        path = Path(metric_csv_path)
        if path.exists():
            try:
                df = pd.read_csv(path)
                if "time" not in df.columns:
                    load_notes.append("CSV missing 'time' column")
                else:
                    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
                    df = df.dropna(subset=["time"]).sort_values("time")
                    value_cols = [c for c in df.columns if c != "time"]
                    if value_cols:
                        for c in value_cols:
                            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
                        pivot = df.set_index("time")[value_cols]
                        pivot.columns = [str(c) for c in pivot.columns]
                        return pivot, "csv", load_notes
                    load_notes.append("CSV has no metric columns")
            except Exception as exc:
                load_notes.append(f"CSV load failed: {exc}")

    return pd.DataFrame(), "none", load_notes


def load_flip_events(path):
    p = Path(path)
    if not p.exists():
        return pd.DataFrame(columns=["timestamp", "to_version", "from_version", "action"])
    with p.open(encoding="utf-8") as handle:
        payload = json.load(handle)

    rows = []
    for item in payload:
        ts = parse_iso_utc(item.get("timestamp"))
        if pd.isna(ts):
            continue
        rows.append({
            "timestamp": ts,
            "to_version": item.get("to_version"),
            "from_version": item.get("from_version"),
            "action": item.get("action", "apply"),
        })

    if not rows:
        return pd.DataFrame(columns=["timestamp", "to_version", "from_version", "action"])

    return pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)


def _first_consecutive_index(bool_series, needed=1):
    if len(bool_series) < needed:
        return None
    arr = np.asarray(bool_series, dtype=bool)
    for i in range(0, len(arr) - needed + 1):
        if arr[i : i + needed].all():
            return i
    return None


def build_run_flip_features(
    run_id,
    pivot,
    flip_df,
    inactive_drop_threshold,
    effective_consecutive_samples,
    baseline_pre_seconds,
    post_flip_window_seconds,
    recovery_target_ratio,
    stabilization_seconds,
):
    flip_rows = []
    qc_rows = []

    if pivot.empty:
        qc_rows.append({"run_id": run_id, "included_for_stats": False, "reason": "Empty metrics timeseries"})
        return pd.DataFrame(), qc_rows

    apply_df = flip_df[flip_df["action"] == "apply"].copy().sort_values("timestamp").reset_index(drop=True)
    if apply_df.empty:
        qc_rows.append({"run_id": run_id, "included_for_stats": False, "reason": "No apply flip events"})
        return pd.DataFrame(), qc_rows

    local = pivot.copy()

    for i, event in apply_df.iterrows():
        apply_time = event["timestamp"]
        next_apply_time = apply_df.loc[i + 1, "timestamp"] if i + 1 < len(apply_df) else (local.index.max() + pd.Timedelta(seconds=1))

        active_version = str(event.get("to_version") or "")
        inactive_version = str(event.get("from_version") or "")
        direction = f"{inactive_version}->{active_version}"

        if active_version not in local.columns:
            local[active_version] = 0.0
        if inactive_version not in local.columns:
            local[inactive_version] = 0.0

        until_next = local[(local.index >= apply_time) & (local.index < next_apply_time)]
        if until_next.empty:
            qc_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": f"Flip {i} has no samples before next apply",
            })
            continue

        inactive_series = until_next[inactive_version].astype(float)
        effective_idx = _first_consecutive_index(
            inactive_series <= inactive_drop_threshold,
            needed=effective_consecutive_samples,
        )
        if effective_idx is None:
            qc_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": f"Flip {i} has no effective switch before next apply",
            })
            continue

        effective_time = inactive_series.index[effective_idx]
        switch_latency_seconds = float((effective_time - apply_time).total_seconds())

        pre_window = local[(local.index >= apply_time - pd.Timedelta(seconds=baseline_pre_seconds)) & (local.index < apply_time)]
        post_window = local[(local.index >= apply_time) & (local.index < apply_time + pd.Timedelta(seconds=post_flip_window_seconds))]

        if pre_window.empty:
            qc_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": f"Flip {i} missing pre-window samples",
            })
            continue

        if post_window.empty:
            qc_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": f"Flip {i} missing post-window samples",
            })
            continue

        pre_total = (pre_window[active_version].astype(float) + pre_window[inactive_version].astype(float))
        post_total = (post_window[active_version].astype(float) + post_window[inactive_version].astype(float))

        baseline_total = float(pre_total.median())
        min_total_post = float(post_total.min())

        if not np.isfinite(baseline_total) or baseline_total <= 0:
            qc_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": f"Flip {i} has non-positive baseline throughput",
            })
            continue

        dip_ratio = max(0.0, float((baseline_total - min_total_post) / baseline_total))

        recovery_target = float(recovery_target_ratio * baseline_total)
        recovery_series = (until_next[active_version].astype(float) + until_next[inactive_version].astype(float))
        rec_idx = _first_consecutive_index(recovery_series >= recovery_target, needed=1)
        recovery_latency_seconds = np.nan
        if rec_idx is not None:
            recovery_time = recovery_series.index[rec_idx]
            recovery_latency_seconds = float((recovery_time - apply_time).total_seconds())

        stable_start = effective_time + pd.Timedelta(seconds=stabilization_seconds)
        stable_window = local[(local.index >= stable_start) & (local.index < next_apply_time)]
        active_throughput_mean = np.nan
        if not stable_window.empty:
            active_throughput_mean = float(stable_window[active_version].astype(float).mean())

        flip_rows.append({
            "run_id": run_id,
            "flip_order": int(i),
            "cycle_index": int(i // 2) + 1,
            "direction": direction,
            "active_version": active_version,
            "inactive_version": inactive_version,
            "flip_apply_timestamp": apply_time,
            "flip_effective_timestamp": effective_time,
            "switch_latency_seconds": switch_latency_seconds,
            "latency_success": bool(switch_latency_seconds <= FLIP_SLA_SECONDS),
            "baseline_total": baseline_total,
            "min_total_post": min_total_post,
            "dip_ratio": dip_ratio,
            "dip_success": bool(dip_ratio <= CONTINUITY_MAX_DIP),
            "recovery_latency_seconds": recovery_latency_seconds,
            "active_throughput_mean": active_throughput_mean,
            "segment_effective_start": effective_time,
            "segment_stabilized_start": stable_start,
            "segment_end": next_apply_time,
        })

    return pd.DataFrame(flip_rows), qc_rows


In [ ]:
# Build run manifest, apply QC, and construct analysis tables
artifacts = discover_artifact_files(DATA_ROOTS)
pairs_df, pair_qc_df = pair_metrics_and_flips(artifacts, tolerance_seconds=PAIR_TOLERANCE_SECONDS)

if artifacts["notes"]:
    print("Discovery notes:")
    for note in artifacts["notes"]:
        print("-", note)

print("Candidate runs discovered:", 0 if pairs_df.empty else len(pairs_df))

run_manifest_rows = []
all_flip_features = []
all_qc_rows = []

if pairs_df.empty:
    run_manifest_df = pd.DataFrame(columns=["run_id", "included_for_stats", "reason"])
    flip_features_df = pd.DataFrame()
else:
    for row in pairs_df.itertuples(index=False):
        run_id = row.run_id
        metric_json_path = row.metric_json_path
        metric_csv_path = row.metric_csv_path
        flip_json_path = row.flip_json_path

        if not row.pair_valid:
            run_manifest_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": "Pairing tolerance failure",
                "metric_source": "none",
                "pair_delta_seconds": row.pair_delta_seconds,
            })
            continue

        pivot, metric_source, load_notes = load_metrics_timeseries(metric_json_path, metric_csv_path)
        flip_df = load_flip_events(flip_json_path)
        apply_df = flip_df[flip_df["action"] == "apply"].copy()

        if load_notes:
            for note in load_notes:
                all_qc_rows.append({"run_id": run_id, "included_for_stats": False, "reason": note})

        if pivot.empty:
            run_manifest_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": "No usable metrics samples",
                "metric_source": metric_source,
                "pair_delta_seconds": row.pair_delta_seconds,
            })
            continue

        if apply_df.empty:
            run_manifest_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": "No apply flip events",
                "metric_source": metric_source,
                "pair_delta_seconds": row.pair_delta_seconds,
            })
            continue

        metric_min = pivot.index.min()
        metric_max = pivot.index.max()
        flip_min = apply_df["timestamp"].min()
        flip_max = apply_df["timestamp"].max()

        overlap_ok = (flip_min >= metric_min) and (flip_max <= metric_max)
        expected_flips_ok = (len(apply_df) == EXPECTED_FLIP_EVENTS)

        if not overlap_ok:
            run_manifest_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": "Flip timestamps not fully contained in metrics range",
                "metric_source": metric_source,
                "pair_delta_seconds": row.pair_delta_seconds,
            })
            continue

        if not expected_flips_ok:
            run_manifest_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": f"Expected {EXPECTED_FLIP_EVENTS} apply flips, found {len(apply_df)}",
                "metric_source": metric_source,
                "pair_delta_seconds": row.pair_delta_seconds,
            })
            continue

        run_flip_df, run_qc = build_run_flip_features(
            run_id=run_id,
            pivot=pivot,
            flip_df=flip_df,
            inactive_drop_threshold=INACTIVE_DROP_THRESHOLD,
            effective_consecutive_samples=EFFECTIVE_CONSECUTIVE_SAMPLES,
            baseline_pre_seconds=BASELINE_PRE_SECONDS,
            post_flip_window_seconds=POST_FLIP_WINDOW_SECONDS,
            recovery_target_ratio=RECOVERY_TARGET_RATIO,
            stabilization_seconds=STABILIZATION_SECONDS,
        )

        all_qc_rows.extend(run_qc)

        if run_flip_df.empty:
            run_manifest_rows.append({
                "run_id": run_id,
                "included_for_stats": False,
                "reason": "No valid flips after QC",
                "metric_source": metric_source,
                "pair_delta_seconds": row.pair_delta_seconds,
            })
            continue

        run_manifest_rows.append({
            "run_id": run_id,
            "included_for_stats": True,
            "reason": "OK",
            "metric_source": metric_source,
            "pair_delta_seconds": row.pair_delta_seconds,
            "metric_start": metric_min,
            "metric_end": metric_max,
            "flip_start": flip_min,
            "flip_end": flip_max,
            "n_apply_flips": len(apply_df),
            "n_valid_flips": len(run_flip_df),
        })

        all_flip_features.append(run_flip_df)

    run_manifest_df = pd.DataFrame(run_manifest_rows)
    flip_features_df = pd.concat(all_flip_features, ignore_index=True) if all_flip_features else pd.DataFrame()

qc_df = pd.concat([
    pair_qc_df if not pair_qc_df.empty else pd.DataFrame(columns=["run_id", "included_for_stats", "reason"]),
    pd.DataFrame(all_qc_rows) if all_qc_rows else pd.DataFrame(columns=["run_id", "included_for_stats", "reason"]),
], ignore_index=True)

print("Runs in manifest:", len(run_manifest_df))
if not run_manifest_df.empty:
    print(run_manifest_df[["run_id", "included_for_stats", "reason"]])
print("Valid flip rows:", 0 if flip_features_df.empty else len(flip_features_df))


In [ ]:
# Statistical tests, run-level aggregation, and synthetic validation scenarios

def wilcoxon_safe(values, alternative="two-sided"):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan, np.nan
    if np.allclose(arr, 0.0):
        return 0.0, 1.0
    try:
        result = wilcoxon(arr, alternative=alternative, zero_method="wilcox", method="auto")
        return float(result.statistic), float(result.pvalue)
    except Exception:
        return np.nan, np.nan


def hodges_lehmann_paired(diffs):
    arr = np.asarray(diffs, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    walsh = (arr[:, None] + arr[None, :]) / 2.0
    return float(np.median(walsh))


def bootstrap_ci(values, stat_fn, n_boot=10000, alpha=0.05, seed=42):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    stats = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(arr, size=arr.size, replace=True)
        stats[i] = stat_fn(sample)
    lo = np.quantile(stats, alpha / 2)
    hi = np.quantile(stats, 1 - alpha / 2)
    return (float(lo), float(hi))


aggregate_rows = []
run_level_rows = []
flip_level_rows = []
synthetic_rows = []

if flip_features_df.empty:
    print("No valid flip features available for statistical testing.")
    run_level_df = pd.DataFrame()
else:
    flip_features_df = flip_features_df.copy()

    run_level_df = (
        flip_features_df.groupby("run_id", as_index=False)
        .agg(
            n_valid_flips=("flip_order", "count"),
            median_latency=("switch_latency_seconds", "median"),
            p95_latency=("switch_latency_seconds", lambda s: float(np.nanpercentile(np.asarray(s, dtype=float), 95))),
            latency_success_rate=("latency_success", lambda s: float(np.mean(np.asarray(s, dtype=float)))),
            median_dip_ratio=("dip_ratio", "median"),
            dip_success_rate=("dip_success", lambda s: float(np.mean(np.asarray(s, dtype=float)))),
            median_recovery_latency=("recovery_latency_seconds", "median"),
        )
    )
    run_level_df["latency_margin_seconds"] = FLIP_SLA_SECONDS - run_level_df["median_latency"]
    run_level_df["continuity_margin_ratio"] = CONTINUITY_MAX_DIP - run_level_df["median_dip_ratio"]

    included_runs = sorted(set(run_level_df["run_id"]))
    n_valid_runs = len(included_runs)
    analysis_mode = "confirmatory" if n_valid_runs >= MIN_RUNS_FOR_CONFIRMATORY else "exploratory"

    # Primary confirmatory tests on run-level margins
    reliability_stat, reliability_p = wilcoxon_safe(run_level_df["latency_margin_seconds"].to_numpy(), alternative="greater")
    continuity_stat, continuity_p = wilcoxon_safe(run_level_df["continuity_margin_ratio"].to_numpy(), alternative="greater")

    primary_names = ["reliability", "continuity"]
    primary_pvals = [reliability_p, continuity_p]
    valid_idx = [i for i, p in enumerate(primary_pvals) if np.isfinite(p)]

    holm_adjusted = {}
    holm_reject = {}
    if valid_idx:
        pvals = [primary_pvals[i] for i in valid_idx]
        reject, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method="holm")
        for local_i, i in enumerate(valid_idx):
            holm_adjusted[primary_names[i]] = float(p_adj[local_i])
            holm_reject[primary_names[i]] = bool(reject[local_i])

    # Flip-level robustness rates with Wilson CI
    n_valid_flips = int(len(flip_features_df))
    latency_success_n = int(flip_features_df["latency_success"].astype(bool).sum())
    dip_success_n = int(flip_features_df["dip_success"].astype(bool).sum())

    latency_prop = (latency_success_n / n_valid_flips) if n_valid_flips > 0 else np.nan
    dip_prop = (dip_success_n / n_valid_flips) if n_valid_flips > 0 else np.nan

    latency_ci_lo, latency_ci_hi = (
        proportion_confint(latency_success_n, n_valid_flips, alpha=ALPHA, method="wilson")
        if n_valid_flips > 0 else (np.nan, np.nan)
    )
    dip_ci_lo, dip_ci_hi = (
        proportion_confint(dip_success_n, n_valid_flips, alpha=ALPHA, method="wilson")
        if n_valid_flips > 0 else (np.nan, np.nan)
    )

    # Secondary exploratory: version difference on stabilized active throughput
    throughput_df = flip_features_df.dropna(subset=["active_throughput_mean"]).copy()
    pair_table = (
        throughput_df.pivot_table(
            index=["run_id", "cycle_index"],
            columns="active_version",
            values="active_throughput_mean",
            aggfunc="mean",
        )
        if not throughput_df.empty else pd.DataFrame()
    )

    if isinstance(pair_table, pd.DataFrame) and ("v1" in pair_table.columns) and ("v2" in pair_table.columns):
        paired_diffs = (pair_table["v2"] - pair_table["v1"]).dropna().to_numpy()
    else:
        paired_diffs = np.array([])

    version_stat, version_p = wilcoxon_safe(paired_diffs, alternative="two-sided")
    hl_estimate = hodges_lehmann_paired(paired_diffs)
    hl_ci_lo, hl_ci_hi = bootstrap_ci(paired_diffs, hodges_lehmann_paired, n_boot=N_BOOT, alpha=ALPHA)

    # Synthetic checks
    synth_pos_latency_margin = np.array([10, 12, 8, 9, 11, 7, 13, 10], dtype=float)
    synth_pos_cont_margin = np.array([0.05, 0.04, 0.03, 0.06, 0.05, 0.04, 0.02, 0.03], dtype=float)
    _, synth_pos_latency_p = wilcoxon_safe(synth_pos_latency_margin, alternative="greater")
    _, synth_pos_cont_p = wilcoxon_safe(synth_pos_cont_margin, alternative="greater")

    synth_neg_latency_margin = np.array([-8, -6, -7, -5, -9, -4, -8, -6], dtype=float)
    synth_neg_cont_margin = np.array([0.01, -0.03, 0.02, -0.04, 0.00, -0.01, 0.01, -0.02], dtype=float)
    _, synth_neg_latency_p = wilcoxon_safe(synth_neg_latency_margin, alternative="greater")
    _, synth_neg_cont_p = wilcoxon_safe(synth_neg_cont_margin, alternative="greater")

    synthetic_rows.extend([
        {
            "scenario": "positive_control_both_primary",
            "expected": "both reject H0",
            "latency_p": synth_pos_latency_p,
            "continuity_p": synth_pos_cont_p,
            "passes_expectation": bool(
                np.isfinite(synth_pos_latency_p) and np.isfinite(synth_pos_cont_p)
                and (synth_pos_latency_p < ALPHA)
                and (synth_pos_cont_p < ALPHA)
            ),
        },
        {
            "scenario": "negative_control_at_least_one_fail",
            "expected": "at least one fails to reject H0",
            "latency_p": synth_neg_latency_p,
            "continuity_p": synth_neg_cont_p,
            "passes_expectation": bool(
                np.isfinite(synth_neg_latency_p) and np.isfinite(synth_neg_cont_p)
                and ((synth_neg_latency_p >= ALPHA) or (synth_neg_cont_p >= ALPHA))
            ),
        },
    ])

    aggregate_rows.extend([
        {"metric": "analysis_mode", "value": analysis_mode},
        {"metric": "alpha", "value": ALPHA},
        {"metric": "target_runs", "value": TARGET_RUNS},
        {"metric": "min_runs_for_confirmatory", "value": MIN_RUNS_FOR_CONFIRMATORY},
        {"metric": "n_valid_runs", "value": n_valid_runs},
        {"metric": "n_valid_flips", "value": n_valid_flips},
        {"metric": "flip_sla_seconds", "value": FLIP_SLA_SECONDS},
        {"metric": "continuity_max_dip", "value": CONTINUITY_MAX_DIP},
        {"metric": "reliability_wilcoxon_stat", "value": reliability_stat},
        {"metric": "reliability_wilcoxon_p_raw", "value": reliability_p},
        {"metric": "reliability_wilcoxon_p_holm", "value": holm_adjusted.get("reliability", np.nan)},
        {"metric": "reliability_reject_holm", "value": holm_reject.get("reliability", False)},
        {"metric": "continuity_wilcoxon_stat", "value": continuity_stat},
        {"metric": "continuity_wilcoxon_p_raw", "value": continuity_p},
        {"metric": "continuity_wilcoxon_p_holm", "value": holm_adjusted.get("continuity", np.nan)},
        {"metric": "continuity_reject_holm", "value": holm_reject.get("continuity", False)},
        {"metric": "latency_success_proportion", "value": latency_prop},
        {"metric": "latency_success_wilson_ci_low", "value": latency_ci_lo},
        {"metric": "latency_success_wilson_ci_high", "value": latency_ci_hi},
        {"metric": "dip_success_proportion", "value": dip_prop},
        {"metric": "dip_success_wilson_ci_low", "value": dip_ci_lo},
        {"metric": "dip_success_wilson_ci_high", "value": dip_ci_hi},
        {"metric": "run_median_latency_median", "value": float(run_level_df["median_latency"].median())},
        {"metric": "run_median_dip_median", "value": float(run_level_df["median_dip_ratio"].median())},
        {"metric": "secondary_n_paired_cycle_diffs", "value": int(len(paired_diffs))},
        {"metric": "secondary_version_wilcoxon_stat", "value": version_stat},
        {"metric": "secondary_version_wilcoxon_p_raw", "value": version_p},
        {"metric": "secondary_version_hodges_lehmann_v2_minus_v1", "value": hl_estimate},
        {"metric": "secondary_version_hl_ci_low", "value": hl_ci_lo},
        {"metric": "secondary_version_hl_ci_high", "value": hl_ci_hi},
    ])

    run_view = run_manifest_df.merge(run_level_df, on="run_id", how="left") if not run_manifest_df.empty else run_level_df
    run_level_rows = run_view.to_dict("records")
    flip_level_rows = flip_features_df.to_dict("records")

aggregate_results_df = pd.DataFrame(aggregate_rows)
run_level_results_df = pd.DataFrame(run_level_rows)
flip_level_results_df = pd.DataFrame(flip_level_rows)
synthetic_results_df = pd.DataFrame(synthetic_rows)

print("Aggregate result rows:", len(aggregate_results_df))
print("Run-level result rows:", len(run_level_results_df))
print("Flip-level result rows:", len(flip_level_results_df))
print("Synthetic check rows:", len(synthetic_results_df))


In [ ]:
# Output artifacts: stat_results CSV, routing_and_latency HTML, statistical_conclusion Markdown
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stamp = ANALYSIS_TIMESTAMP_UTC
stat_results_path = OUTPUT_DIR / f"stat_results_{stamp}.csv"
plot_html_path = OUTPUT_DIR / f"routing_and_latency_{stamp}.html"
conclusion_md_path = OUTPUT_DIR / f"statistical_conclusion_{stamp}.md"

# stat_results CSV (aggregate + run-level + flip-level + synthetic checks)
parts = []
if not aggregate_results_df.empty:
    tmp = aggregate_results_df.copy()
    tmp.insert(0, "section", "aggregate")
    parts.append(tmp)
if not run_level_results_df.empty:
    tmp = run_level_results_df.copy()
    tmp.insert(0, "section", "run_level")
    parts.append(tmp)
if not flip_level_results_df.empty:
    tmp = flip_level_results_df.copy()
    tmp.insert(0, "section", "flip_level")
    parts.append(tmp)
if not synthetic_results_df.empty:
    tmp = synthetic_results_df.copy()
    tmp.insert(0, "section", "synthetic_checks")
    parts.append(tmp)

if parts:
    stat_results_df = pd.concat(parts, ignore_index=True, sort=False)
else:
    stat_results_df = pd.DataFrame(columns=["section", "metric", "value"])

stat_results_df.to_csv(stat_results_path, index=False)

# routing_and_latency HTML
if not flip_level_results_df.empty:
    fig = make_subplots(
        rows=2,
        cols=2,
        shared_xaxes=False,
        vertical_spacing=0.13,
        subplot_titles=(
            "Switch Latency by Direction",
            "Throughput Dip Ratio by Direction",
            "Run-Level Median Latency",
            "Run-Level Median Dip Ratio",
        ),
    )

    for direction in sorted(flip_level_results_df["direction"].dropna().unique()):
        sub = flip_level_results_df[flip_level_results_df["direction"] == direction]
        fig.add_trace(
            go.Box(
                y=sub["switch_latency_seconds"],
                name=f"lat {direction}",
                boxmean=True,
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Box(
                y=sub["dip_ratio"],
                name=f"dip {direction}",
                boxmean=True,
            ),
            row=1,
            col=2,
        )

    fig.add_hline(y=FLIP_SLA_SECONDS, row=1, col=1, line_dash="dash", line_color="red")
    fig.add_hline(y=CONTINUITY_MAX_DIP, row=1, col=2, line_dash="dash", line_color="red")

    if not run_level_results_df.empty and ("median_latency" in run_level_results_df.columns):
        run_plot = run_level_results_df.dropna(subset=["median_latency", "median_dip_ratio"]).copy()
        if not run_plot.empty:
            run_plot = run_plot.sort_values("run_id").reset_index(drop=True)
            run_plot["run_order"] = np.arange(1, len(run_plot) + 1)

            fig.add_trace(
                go.Scatter(
                    x=run_plot["run_order"],
                    y=run_plot["median_latency"],
                    mode="lines+markers",
                    name="run median latency",
                ),
                row=2,
                col=1,
            )
            fig.add_hline(y=FLIP_SLA_SECONDS, row=2, col=1, line_dash="dash", line_color="red")

            fig.add_trace(
                go.Scatter(
                    x=run_plot["run_order"],
                    y=run_plot["median_dip_ratio"],
                    mode="lines+markers",
                    name="run median dip",
                ),
                row=2,
                col=2,
            )
            fig.add_hline(y=CONTINUITY_MAX_DIP, row=2, col=2, line_dash="dash", line_color="red")

    fig.update_layout(
        title="Transition Reliability and Service Continuity",
        template="plotly_white",
        height=950,
        showlegend=True,
    )
    fig.update_yaxes(title_text="Seconds", row=1, col=1)
    fig.update_yaxes(title_text="Dip ratio", row=1, col=2)
    fig.update_yaxes(title_text="Seconds", row=2, col=1)
    fig.update_yaxes(title_text="Dip ratio", row=2, col=2)
    fig.update_xaxes(title_text="Run order", row=2, col=1)
    fig.update_xaxes(title_text="Run order", row=2, col=2)

    fig.write_html(plot_html_path, include_plotlyjs="cdn", full_html=True)

# statistical_conclusion Markdown
analysis_mode = "exploratory"
n_valid_runs = 0
n_valid_flips = 0
reliability_p_holm = np.nan
continuity_p_holm = np.nan
reliability_reject = False
continuity_reject = False
latency_prop = np.nan
latency_ci_low = np.nan
latency_ci_high = np.nan
dip_prop = np.nan
dip_ci_low = np.nan
dip_ci_high = np.nan
secondary_p = np.nan

if not aggregate_results_df.empty:
    agg_map = {str(r["metric"]): r["value"] for r in aggregate_results_df.to_dict("records")}
    analysis_mode = str(agg_map.get("analysis_mode", "exploratory"))
    n_valid_runs = int(float(agg_map.get("n_valid_runs", 0)))
    n_valid_flips = int(float(agg_map.get("n_valid_flips", 0)))

    reliability_p_holm = agg_map.get("reliability_wilcoxon_p_holm", np.nan)
    continuity_p_holm = agg_map.get("continuity_wilcoxon_p_holm", np.nan)
    reliability_reject = bool(agg_map.get("reliability_reject_holm", False))
    continuity_reject = bool(agg_map.get("continuity_reject_holm", False))

    latency_prop = agg_map.get("latency_success_proportion", np.nan)
    latency_ci_low = agg_map.get("latency_success_wilson_ci_low", np.nan)
    latency_ci_high = agg_map.get("latency_success_wilson_ci_high", np.nan)

    dip_prop = agg_map.get("dip_success_proportion", np.nan)
    dip_ci_low = agg_map.get("dip_success_wilson_ci_low", np.nan)
    dip_ci_high = agg_map.get("dip_success_wilson_ci_high", np.nan)

    secondary_p = agg_map.get("secondary_version_wilcoxon_p_raw", np.nan)

reliability_statement = (
    "Transition reliability claim is statistically supported (Holm-adjusted)."
    if reliability_reject else
    "Transition reliability claim is not statistically supported at configured SLA."
)
continuity_statement = (
    "Service continuity claim is statistically supported (Holm-adjusted)."
    if continuity_reject else
    "Service continuity claim is not statistically supported at configured dip threshold."
)

conclusion_lines = [
    "# Statistical Conclusion for Time-Shifted A/B Testing",
    "",
    f"- Analysis timestamp (UTC): {stamp}",
    f"- Mode: **{analysis_mode}** (valid runs: {n_valid_runs}, target runs: {TARGET_RUNS})",
    f"- Alpha: {ALPHA}",
    f"- Effective switch definition: inactive throughput <= {INACTIVE_DROP_THRESHOLD}",
    f"- Flip latency SLA (s): {FLIP_SLA_SECONDS}",
    f"- Continuity max dip ratio: {CONTINUITY_MAX_DIP}",
    "",
    "## Primary hypothesis decisions",
    f"1. Transition reliability (one-sided Wilcoxon on run-level SLA margin): p_holm = {reliability_p_holm}",
    f"   - {reliability_statement}",
    f"2. Service continuity (one-sided Wilcoxon on run-level dip margin): p_holm = {continuity_p_holm}",
    f"   - {continuity_statement}",
    "",
    "## Robustness summaries (flip-level)",
    f"- Valid flips included: {n_valid_flips}",
    f"- P(latency <= {FLIP_SLA_SECONDS}s): {latency_prop} (Wilson CI: {latency_ci_low}, {latency_ci_high})",
    f"- P(dip <= {CONTINUITY_MAX_DIP}): {dip_prop} (Wilson CI: {dip_ci_low}, {dip_ci_high})",
    "",
    "## Secondary exploratory analysis",
    f"- Version throughput difference (two-sided Wilcoxon, raw p): {secondary_p}",
    "",
    "## Interpretation guidance",
    "- If mode is exploratory, treat results as preliminary.",
    "- Confirmatory interpretation requires at least MIN_RUNS_FOR_CONFIRMATORY valid runs.",
]

conclusion_md_path.write_text("\n".join(conclusion_lines), encoding="utf-8")

print("Wrote:")
print("-", stat_results_path)
print("-", plot_html_path if plot_html_path.exists() else f"{plot_html_path} (not generated: no valid flips)")
print("-", conclusion_md_path)

if not stat_results_df.empty:
    display(stat_results_df.head(20))
